# 🚨 Análise de Fraudes - Streaming

Este notebook contém a análise exploratória dos dados de fraudes detectadas pelo sistema de streaming.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Bibliotecas carregadas com sucesso!')

## 1. Geração de Dados Simulados

Vamos gerar dados simulados para análise, já que o sistema de streaming ainda não está em produção.

In [ ]:
np.random.seed(42)
n_transacoes = 10000
n_fraudulentas = int(n_transacoes * 0.12)
n_normais = n_transacoes - n_fraudulentas

# Gerar transações normais
transacoes_normais = pd.DataFrame({
    'transaction_id': [f'TXN-{i:06d}' for i in range(n_normais)],
    'user_id': np.random.randint(1, 1000, n_normais),
    'valor': np.random.exponential(500, n_normais).clip(10, 5000),
    'timestamp': [datetime.now() - timedelta(minutes=np.random.randint(0, 1440)) for _ in range(n_normais)],
    'pais': np.random.choice(['BR', 'BR', 'BR', 'BR', 'US'], n_normais),
    'dispositivo': np.random.choice(['mobile', 'desktop', 'tablet'], n_normais, p=[0.6, 0.3, 0.1]),
    'categoria': np.random.choice(['alimentacao', 'transporte', 'saude', 'compras', 'lazer'], n_normais),
    'is_fraud': False
})

# Gerar transações fraudulentas
transacoes_fraudulentas = pd.DataFrame({
    'transaction_id': [f'FRD-{i:06d}' for i in range(n_fraudulentas)],
    'user_id': np.random.randint(1, 1000, n_fraudulentas),
    'valor': np.random.exponential(5000, n_fraudulentas).clip(1000, 50000),
    'timestamp': [datetime.now() - timedelta(minutes=np.random.randint(0, 1440)) for _ in range(n_fraudulentas)],
    'pais': np.random.choice(['BR', 'US', 'GB', 'JP'], n_fraudulentas),
    'dispositivo': np.random.choice(['mobile', 'desktop', 'atm', 'pos'], n_fraudulentas),
    'categoria': np.random.choice(['transferencia', 'saque', 'compras'], n_fraudulentas),
    'is_fraud': True
})

# Combinar
df = pd.concat([transacoes_normais, transacoes_fraudulentas], ignore_index=True)
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6])

print(f'📊 Total de transações: {len(df)}')
print(f'🚨 Fraudulentas: {df.is_fraud.sum()} ({df.is_fraud.mean()*100:.1f}%)')
print(f'✅ Normais: {(~df.is_fraud).sum()} ({(~df.is_fraud).mean()*100:.1f}%)')

## 2. Análise Exploratória

In [ ]:
# Distribuição de valores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[~df.is_fraud]['valor'].hist(bins=50, ax=axes[0], alpha=0.7, color='green', label='Normal')
axes[0].set_title('Distribuição de Valores - Transações Normais')
axes[0].set_xlabel('Valor (R$)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

df[df.is_fraud]['valor'].hist(bins=50, ax=axes[1], alpha=0.7, color='red', label='Fraude')
axes[1].set_title('Distribuição de Valores - Transações Fraudulentas')
axes[1].set_xlabel('Valor (R$)')
axes[1].set_ylabel('Frequência')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Taxa de fraude por hora
fraude_por_hora = df.groupby('hour')['is_fraud'].mean() * 100

plt.figure(figsize=(12, 5))
fraude_por_hora.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Taxa de Fraude por Hora do Dia')
plt.xlabel('Hora')
plt.ylabel('Taxa de Fraude (%)')
plt.xticks(rotation=0)
plt.axhline(y=df['is_fraud'].mean()*100, color='red', linestyle='--', label='Média Geral')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fraudes por país
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('pais')['is_fraud'].sum().sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Total de Fraudes por País')
axes[0].set_xlabel('Quantidade')

df.groupby('pais')['is_fraud'].mean().sort_values(ascending=True).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Taxa de Fraude por País')
axes[1].set_xlabel('Taxa (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Fraudes por dispositivo
plt.figure(figsize=(10, 5))
pd.crosstab(df['dispositivo'], df['is_fraud'], normalize='index').plot(kind='bar', stacked=True)
plt.title('Proporção de Fraudes por Dispositivo')
plt.xlabel('Dispositivo')
plt.ylabel('Proporção')
plt.legend(['Normal', 'Fraude'], loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação
df_encoded = df.copy()
df_encoded['dispositivo_encoded'] = df_encoded['dispositivo'].astype('category').cat.codes
df_encoded['categoria_encoded'] = df_encoded['categoria'].astype('category').cat.codes
df_encoded['pais_encoded'] = df_encoded['pais'].astype('category').cat.codes

cols_corr = ['valor', 'hour', 'day_of_week', 'is_weekend', 'dispositivo_encoded', 'categoria_encoded', 'pais_encoded', 'is_fraud']
corr = df_encoded[cols_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

## 3. Estatísticas Descritivas

In [ ]:
# Comparação entre transações normais e fraudulentas
print('='*60)
print('Estatísticas Comparativas')
print('='*60)

for col in ['valor']:
    print(f'\n{col.upper()}:')
    print(f'  Normal  -> Média: R$ {df[~df.is_fraud][col].mean():.2f}, Mediana: R$ {df[~df.is_fraud][col].median():.2f}')
    print(f'  Fraude  -> Média: R$ {df[df.is_fraud][col].mean():.2f}, Mediana: R$ {df[df.is_fraud][col].median():.2f}')

print(f'\nHorário médio:')
print(f'  Normal  -> {df[~df.is_fraud]["hour"].mean():.1f}h')
print(f'  Fraude  -> {df[df.is_fraud]["hour"].mean():.1f}h')

print(f'\nProporção de fim de semana:')
print(f'  Normal  -> {df[~df.is_fraud]["is_weekend"].mean()*100:.1f}%')
print(f'  Fraude  -> {df[df.is_fraud]["is_weekend"].mean()*100:.1f}%')

## 4. Regras de Detecção

Teste das regras de detecção de fraudes.

In [ ]:
# Regra 1: Valor alto
limiar_valor = 10000
df['regra_valor'] = df['valor'] > limiar_valor

# Regra 2: Horário incomum (00:00 - 05:00)
df['regra_horario'] = df['hour'].between(0, 5)

# Regra 3: País de risco
paises_risco = ['US', 'GB', 'JP']
df['regra_pais'] = df['pais'].isin(paises_risco)

# Combinação de regras
df['regras_ativadas'] = df[['regra_valor', 'regra_horario', 'regra_pais']].sum(axis=1)
df['alerta_regra'] = df['regras_ativadas'] >= 2

print(f'\n🔍 Resultado das Regras:')
print(f'  Regra Valor Ativada: {df.regra_valor.sum()} transações')
print(f'  Regra Horário Ativada: {df.regra_horario.sum()} transações')
print(f'  Regra País Ativada: {df.regra_pais.sum()} transações')
print(f'  Alertas (≥2 regras): {df.alerta_regra.sum()} transações')

# Precisão das regras
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = df['is_fraud']
y_pred = df['alerta_regra']

print(f'\n📊 Métricas das Regras:')
print(f'  Precisão: {precision_score(y_true, y_pred):.2%}')
print(f'  Recall: {recall_score(y_true, y_pred):.2%}')
print(f'  F1-Score: {f1_score(y_true, y_pred):.2%}')

## 5. Resumo

### Principais Descobertas

1. **Valores**: Transações fraudulentas tendem a ter valores significativamente maiores
2. **Horário**: Fraudes são mais frequentes durante a madrugada (00:00 - 05:00)
3. **País**: Transações internacionais apresentam maior risco de fraude
4. **Dispositivo**: Dispositivos ATM e POS estão associados a mais fraudes

### Próximos Passos

- Implementar modelo de ML para detecção mais precisa
- Integrar com pipeline de streaming em tempo real
- Configurar alertas no Grafana